**Name:  Anjali Arvind Shirke**    
**Roll No : 07**    
**Assignment 8**   

### Aim
Write a python program to illustrate ART (Adaptive Resonance Theory) neural network.

### Step 1: Import Required Packages

In [1]:
import numpy as np

### Step 2: Define ART1 Model Class
The ART1 class implements:
- `_choice(x, w)` — Computes choice function (how well a category matches the input).
- `_match(x, w)` — Computes match function (ratio of overlap to input norm).
- `train(data)` — Assigns each pattern to a category or creates a new one.
- `predict(x)` — Returns the best matching category for a given input.

In [2]:
class ART1:
    def __init__(self, vigilance=0.8):
        self.vigilance = vigilance
        self.weights = []

    def _choice(self, x, w):
        """Choice function: measures how well weight w matches input x."""
        return np.sum(np.minimum(x, w)) / (0.0001 + np.sum(w))

    def _match(self, x, w):
        """Match function: checks if the match meets vigilance threshold."""
        return np.sum(np.minimum(x, w)) / (0.0001 + np.sum(x))

    def train(self, data):
        """Train the ART1 network on a set of binary patterns."""
        for x in data:
            if len(self.weights) == 0:
                # First pattern — create the first category
                self.weights.append(x.copy())
                continue

            # Find the best matching category
            best_j = -1
            best_choice = -1
            for j, w in enumerate(self.weights):
                choice_val = self._choice(x, w)
                if choice_val > best_choice:
                    best_choice = choice_val
                    best_j = j

            # Check if the match meets the vigilance criterion
            if self._match(x, self.weights[best_j]) >= self.vigilance:
                # Update the category weight (AND operation)
                self.weights[best_j] = np.minimum(x, self.weights[best_j])
            else:
                # Create a new category
                self.weights.append(x.copy())

    def predict(self, x):
        """Predict the best matching category for input x."""
        scores = [self._choice(x, w) for w in self.weights]
        return np.argmax(scores)

* **Choice Function (_choice(x, w)):** Measures how well category w matches input x.  
Formula: choice = sum(min(x_i, w_i) for i in features) / (epsilon + sum(w_i))  
epsilon = 0.0001 prevents division by zero.  
Intuition: Higher if many "on" bits in x are also "on" in w, normalized by w's size. E.g., if x and w are identical, choice = 1.0.  

* **Match Function (_match(x, w)):** Checks similarity as a fraction of the input's "on" bits that overlap with the category.  
Formula: match = sum(min(x_i, w_i) for i in features) / (epsilon + sum(x_i))  
Intuition: Ratio of overlapping "on" bits to total "on" bits in x. E.g., if x has 2 "on" bits and both match w, match = 1.0. Vigilance (0.75) is the threshold—if match < 0.75, no resonance (create new category).  

### Step 3: Define Binary Input Patterns

In [3]:
# Binary patterns with 4 features
patterns = np.array([
    [1, 0, 1, 0],
    [1, 0, 1, 1],
    [0, 1, 0, 1],
    [0, 1, 0, 0],
])

print('Input Patterns:')
for i, p in enumerate(patterns):
    print(f'  Pattern {i}: {p}')

Input Patterns:
  Pattern 0: [1 0 1 0]
  Pattern 1: [1 0 1 1]
  Pattern 2: [0 1 0 1]
  Pattern 3: [0 1 0 0]


### Step 4: Create and Train ART1 Model

In [4]:
art = ART1(vigilance=0.75)
art.train(patterns)

print('Vigilance parameter:', art.vigilance)
print('Number of categories created:', len(art.weights))
print()
print('Category weights:')
for j, w in enumerate(art.weights):
    print(f'  Category {j}: {w}')

Vigilance parameter: 0.75
Number of categories created: 3

Category weights:
  Category 0: [1 0 1 0]
  Category 1: [1 0 1 1]
  Category 2: [0 1 0 0]


### Step 5: Predict Categories for Each Pattern

In [5]:
print('\nCategory assignments:')
for i, p in enumerate(patterns):
    category = art.predict(p)
    print(f'  Pattern {i} {p} -> Category {category}')


Category assignments:
  Pattern 0 [1 0 1 0] -> Category 0
  Pattern 1 [1 0 1 1] -> Category 1
  Pattern 2 [0 1 0 1] -> Category 2
  Pattern 3 [0 1 0 0] -> Category 2


### Step 6: Test with a New (Unseen) Pattern

In [6]:
new_pattern = np.array([1, 0, 0, 0])
predicted_category = art.predict(new_pattern)
print(f'New pattern {new_pattern} -> Category {predicted_category}')

New pattern [1 0 0 0] -> Category 0
